In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np

from grf_pipeline_utils.data_utils import ad2float, interp_segments
from grf_pipeline_utils.opensim_utils import *
from grf_pipeline_utils.signal_processing import *

In [ ]:
root_dir = '/Users/briankeller/Desktop/GRFMuscleModel/Old_Young_Walking_Data/'
with open(root_dir + 'all_segs.pkl', 'rb') as f:
    all_segs = pickle.load(f)
print(all_segs)

# Batch Plotting Inverse Kinematics Segments

In [ ]:
def ik_data_to_segs(
    angles,
    seg_times,
    problem_trials,
    ik_data_dir,
    ik_suffix="_ik_filtered.mot",
):

    compiled_segs = {}

    # base angles in-order, allow both suffixed and unsuffixed inputs
    base_angles = []
    seen = set()
    for a in angles:
        b = a[:-2] if a.endswith(("_r", "_l")) else a
        if b not in seen:
            base_angles.append(b)
            seen.add(b)

    def load_ik_columns(storage, base_angles, side_suffix):
        time_col = osim.ArrayDouble()
        storage.getTimeColumn(time_col)
        t = ad2float(time_col)

        data = {}
        UNSUFFIXED = {
        "pelvis_tilt","pelvis_list","pelvis_rotation",
        "pelvis_tx","pelvis_ty","pelvis_tz",
        "lumbar_extension","lumbar_bending","lumbar_rotation"
    }

        for a in base_angles:
            col = osim.ArrayDouble()
            if a in UNSUFFIXED:
                storage.getDataColumn(a, col)
            else:
                storage.getDataColumn(f"{a}_{side_suffix}", col)
            data[a] = ad2float(col)

        return t, data

    problematic_segs = []

    for subject, trials in seg_times.items():
        compiled_segs[subject] = {a: [] for a in base_angles}

        for trial_name, seg_dict in trials.items():
            if trial_name in problem_trials:
                continue

            ik_path = os.path.join(ik_data_dir, f"{trial_name}{ik_suffix}")
            ik_storage = osim.Storage(ik_path)

            time, ik_r = load_ik_columns(ik_storage, base_angles, "r")
            _,    ik_l = load_ik_columns(ik_storage, base_angles, "l")

            for side, seg_list in seg_dict.items():
                side = side.lower()
                if side not in ("right", "left"):
                    continue

                ik_data = ik_r if side == "right" else ik_l

                for (s, e) in seg_list:
                    mask = (time >= s) & (time <= e)
                    if not mask.any():
                        continue
                    # ankle_seg = ik_data['ankle_angle'][mask]
                    # ankle_idx_75 = int(len(ankle_seg) * 0.75)
                    # if ankle_seg[ankle_idx_75] < -5:
                    #     problematic_segs.append({
                    #         'subject': trial_name,
                    #         'side': side,
                    #         'start_time': s,
                    #         'end_time':float(e)
                    #     })
                    #     continue
                    
                    for a in base_angles:
                        compiled_segs[subject][a].append(ik_data[a][mask])

    return compiled_segs, problematic_segs

In [ ]:
angles = [
"time",
    "pelvis_tilt",
    "pelvis_list",
    "pelvis_rotation",
    "pelvis_tx",
    "pelvis_ty",
    "pelvis_tz",
    "hip_flexion_r",
    "hip_adduction_r",
    "hip_rotation_r",
    "knee_angle_r",
    "knee_angle_r_beta",
    "ankle_angle_r",
    "subtalar_angle_r",
    "mtp_angle_r",
    "hip_flexion_l",
    "hip_adduction_l",
    "hip_rotation_l",
    "knee_angle_l",
    "knee_angle_l_beta",
    "ankle_angle_l",
    "mtp_angle_l"
]

ik_segs, ik_problem = ik_data_to_segs(
    angles=angles,
    seg_times=all_segs,
    problem_trials=[],
    ik_data_dir="/Users/briankeller/Desktop/GRFMuscleModel/Old_Young_Walking_Data/Results/IK/filtered"
)

In [ ]:
for seg in ik_problem:
    trial = seg.get('subject')
    time = seg.get('start_time')
    print(f'Trial:{trial}, start time:{time}')

In [ ]:
n_interp_points = 100
ik_resampled = {}
time_resampled = None

for subj, subj_data in ik_segs.items():
    ik_resampled[subj] = {}
    for key, seg_list in subj_data.items():
        if len(seg_list) == 0:
            ik_resampled[subj][key] = []
            continue

        resampled, t = interp_segments(seg_list, n_interp_points)
        ik_resampled[subj][key] = resampled

        # grab time_resampled once (they should all be identical if interp_segments is consistent)
        if time_resampled is None:
            time_resampled = t

# store once at the end
ik_resampled["time_resampled"] = time_resampled

def get_all_segments(resampled_segs, key):
    """
    Collects all resampled segments for a given signal across all subjects.
    """
    all_segs = []
    for subj, data in resampled_segs.items():
        if subj == "time_resampled":
            continue
        if key in data:
            all_segs.extend(data[key])
    return np.array(all_segs)

In [ ]:
ik_keys = [
    "pelvis_tilt",
    "pelvis_list",
    "pelvis_rotation",
    "pelvis_tx",
    "pelvis_ty",
    "pelvis_tz",
    "hip_flexion",
    "hip_adduction",
    "hip_rotation",
    "knee_angle",
    "ankle_angle",
    "subtalar_angle",
    "mtp_angle"
]

# compile across subjects ONCE (fast + consistent)
resampled_angles = {k: get_all_segments(ik_resampled, k) for k in ik_keys}
no_convert = {"pelvis_tx", "pelvis_ty", "pelvis_tz", "time"}

# fix time axis once (handles (T,) or (N,T))
t = np.asarray(ik_resampled["time_resampled"])
time_axis = t[0] if t.ndim == 2 else t   # (T,)

In [ ]:
def plot_ik_grid_compiled(
    resampled_angles,
    time_axis,
    angle_keys,
    nrows=5,
    ncols=3,
    figsize=(15, 10),
    alpha=0.25,
    linewidth=1.5,
    plot_mean=True,
    mean_linewidth=3,
):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten()

    x = time_axis * 100  # percent stance, (T,)

    for i, ax in enumerate(axes):
        if i >= len(angle_keys):
            ax.axis("off")
            continue

        key = angle_keys[i]
        segments = resampled_angles.get(key, None)
        if segments is None or len(segments) == 0:
            ax.set_title(key.replace("_", " ").title(), fontsize=14)
            ax.text(0.5, 0.5, "No segments", ha="center", va="center", transform=ax.transAxes)
            ax.grid(True, alpha=0.3)
            continue

        # segments should be (N,T) if get_all_segments returns np.array
        for seg in segments:
            ax.plot(x, seg, alpha=alpha, linewidth=linewidth)

        if plot_mean:
            Y = np.asarray(segments)
            if Y.ndim == 2:
                ax.plot(x, Y.mean(axis=0), linewidth=mean_linewidth)

        if i >= (nrows - 1) * ncols:
            ax.set_xlabel("Percent Normalized Stance", fontsize=12)
        if i % ncols == 0:
            ax.set_ylabel("Value", fontsize=12)

        ax.set_title(key.replace("_", " ").title(), fontsize=14)
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis="x", labelsize=12)
        ax.tick_params(axis="y", labelsize=12)

    plt.tight_layout()

In [ ]:
plot_ik_grid_compiled(resampled_angles, time_axis, ik_keys, nrows=5, ncols=3)

# Batch Plot Inverse Dynamics

In [ ]:
def id_data_to_segs(
    measures,
    seg_times,
    problem_trials,
    id_data_dir,
    id_suffix="_id.sto",   # adjust to your actual filename pattern
):
    compiled_segs = {}

    # base measures in-order (drop _r/_l before _moment/_force if present)
    base_measures = []
    seen = set()
    for m in measures:
        if m.endswith(("_r_moment", "_l_moment")):
            b = m.replace("_r_moment", "").replace("_l_moment", "") + "_moment"
        elif m.endswith(("_r_force", "_l_force")):
            b = m.replace("_r_force", "").replace("_l_force", "") + "_force"
        else:
            b = m  # pelvis/lumbar already unsuffixed

        if b not in seen:
            base_measures.append(b)
            seen.add(b)

    # which columns are unsuffixed in the file
    UNSUFFIXED = {
        "pelvis_tilt_moment", "pelvis_list_moment", "pelvis_rotation_moment",
        "pelvis_tx_force", "pelvis_ty_force", "pelvis_tz_force",
        "lumbar_extension_moment", "lumbar_bending_moment", "lumbar_rotation_moment",
    }

    def load_id_columns(storage, base_measures, side_suffix):
        time_col = osim.ArrayDouble()
        storage.getTimeColumn(time_col)
        t = ad2float(time_col)

        data = {}
        for b in base_measures:
            col = osim.ArrayDouble()

            if b in UNSUFFIXED:
                storage.getDataColumn(b, col)
            else:
                # b looks like "hip_flexion_moment" or "knee_angle_force"
                # file columns look like "hip_flexion_r_moment"
                if b.endswith("_moment"):
                    name = b.replace("_moment", f"_{side_suffix}_moment")
                elif b.endswith("_force"):
                    name = b.replace("_force", f"_{side_suffix}_force")
                else:
                    name = f"{b}_{side_suffix}"  # fallback (shouldn't really happen)
                storage.getDataColumn(name, col)

            data[b] = ad2float(col)

        return t, data

    problematic_segs = []

    for subject, trials in seg_times.items():
        compiled_segs[subject] = {b: [] for b in base_measures}

        for trial_name, seg_dict in trials.items():
            if trial_name in problem_trials:
                continue

            id_path = os.path.join(id_data_dir, f"{trial_name}{id_suffix}")
            id_storage = osim.Storage(id_path)

            time, id_r = load_id_columns(id_storage, base_measures, "r")
            _,    id_l = load_id_columns(id_storage, base_measures, "l")

            for side, seg_list in seg_dict.items():
                side = side.lower()
                if side not in ("right", "left"):
                    continue

                id_data = id_r if side == "right" else id_l

                for (s, e) in seg_list:
                    mask = (time >= s) & (time <= e)
                    if not mask.any():
                        continue

                    for b in base_measures:
                        compiled_segs[subject][b].append(id_data[b][mask])

    return compiled_segs, problematic_segs

In [ ]:
id_measures = [
    "pelvis_tilt_moment","pelvis_list_moment","pelvis_rotation_moment",
    "pelvis_tx_force","pelvis_ty_force","pelvis_tz_force",
    "hip_flexion_r_moment","hip_adduction_r_moment","hip_rotation_r_moment",
    "hip_flexion_l_moment","hip_adduction_l_moment","hip_rotation_l_moment",
    "lumbar_extension_moment","lumbar_bending_moment","lumbar_rotation_moment",
    "knee_angle_r_moment","knee_angle_r_beta_force",
    "knee_angle_l_moment","knee_angle_l_beta_force",
    "ankle_angle_r_moment","ankle_angle_l_moment",
    "subtalar_angle_r_moment","subtalar_angle_l_moment",
    "mtp_angle_r_moment","mtp_angle_l_moment"
]

id_segs, id_problem = id_data_to_segs(
    measures=id_measures,
    seg_times=all_segs,
    problem_trials=[],
    id_data_dir="/Users/briankeller/Desktop/GRFMuscleModel/Old_Young_Walking_Data/Results/ID/filtered",  # example
    id_suffix="_id_filtered.mot" 
)

In [ ]:
n_interp_points = 100
id_resampled = {}
time_resampled = None

for subj, subj_data in id_segs.items():
    id_resampled[subj] = {}
    for key, seg_list in subj_data.items():
        if len(seg_list) == 0:
            id_resampled[subj][key] = []
            continue

        resampled, t = interp_segments(seg_list, n_interp_points)
        id_resampled[subj][key] = resampled

        if time_resampled is None:
            t = np.asarray(t)
            time_resampled = t[0] if t.ndim == 2 else t  # force (T,)

id_resampled["time_resampled"] = time_resampled

In [ ]:
id_keys = [
    "hip_flexion_moment", "hip_adduction_moment", "hip_rotation_moment",
    "knee_angle_moment", "ankle_angle_moment",
    "subtalar_angle_moment", "mtp_angle_moment",
    "pelvis_tx_force", "pelvis_ty_force", "pelvis_tz_force",
    "pelvis_tilt_moment", "pelvis_list_moment", "pelvis_rotation_moment",
]

id_compiled = {k: get_all_segments(id_resampled, k) for k in id_keys}
time_axis = id_resampled["time_resampled"]  # (T,)

In [ ]:
def plot_id_grid_compiled(
    id_compiled,
    time_axis,
    keys,
    nrows=4,
    ncols=3,
    figsize=(16, 10),
    alpha=0.25,
    linewidth=1.5,
    plot_mean=True,
    mean_linewidth=3,
):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten()

    x = np.asarray(time_axis).reshape(-1) * 100  # percent stance

    for i, ax in enumerate(axes):
        if i >= len(keys):
            ax.axis("off")
            continue

        k = keys[i]
        segs = id_compiled.get(k, None)

        if segs is None or len(segs) == 0:
            ax.set_title(k.replace("_", " ").title(), fontsize=12)
            ax.text(0.5, 0.5, "No segments", ha="center", va="center", transform=ax.transAxes)
            ax.grid(True, alpha=0.3)
            continue

        for seg in segs:
            ax.plot(x, seg, alpha=alpha, linewidth=linewidth)

        if plot_mean:
            Y = np.asarray(segs)
            if Y.ndim == 2:
                ax.plot(x, Y.mean(axis=0), linewidth=mean_linewidth)

        if i >= (nrows - 1) * ncols:
            ax.set_xlabel("Percent Normalized Stance", fontsize=11)
        if i % ncols == 0:
            ax.set_ylabel("Moment / Force", fontsize=11)

        ax.set_title(k.replace("_", " ").title(), fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis="x", labelsize=10)
        ax.tick_params(axis="y", labelsize=10)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_id_grid_compiled(id_compiled, time_axis, id_keys, nrows=4, ncols=3)

In [ ]:
print(id_compiled.keys())

In [ ]:
print(np.mean(id_compiled['pelvis_ty_force']))

# Batch Plot Activations

In [ ]:
def activations_to_segs(muscles, seg_times, problem_trials, states_dir, states_filename="results_states.sto"):
    """
    muscles: list like ["tibant_r", "tibant_l", ...] or your full muscles list
    states_dir: base dir that contains trial subfolders like muscle_force_dir did
               e.g. .../Results/SO
    reads: <states_dir>/<trial_name>/<states_filename>
    extracts: /forceset/<muscle_side>/activation
    stores under base names (no _r/_l) just like your other segment dicts
    """
    compiled_segs = {}

    # base muscles, in sorted unique order
    base_muscles = sorted({m[:-2] for m in muscles if m.endswith(("_r", "_l"))})

    def load_activation_columns(storage, base_muscles, side_suffix):
        time_col = osim.ArrayDouble()
        storage.getTimeColumn(time_col)
        t = ad2float(time_col)

        data = {}
        for m in base_muscles:
            col = osim.ArrayDouble()
            # states file uses path-style column names:
            storage.getDataColumn(f"/forceset/{m}_{side_suffix}/activation", col)
            data[m] = ad2float(col)
        return t, data

    problematic_segs = []

    for subject, trials in seg_times.items():
        compiled_segs[subject] = {m: [] for m in base_muscles}

        for trial_name, seg_dict in trials.items():
            if trial_name in problem_trials:
                continue

            states_path = os.path.join(states_dir, trial_name, states_filename)
            states_storage = osim.Storage(states_path)

            time, act_r = load_activation_columns(states_storage, base_muscles, "r")
            _,    act_l = load_activation_columns(states_storage, base_muscles, "l")

            for side, seg_list in seg_dict.items():
                side = side.lower()
                if side not in ("right", "left"):
                    continue

                act_data = act_r if side == "right" else act_l

                for (s, e) in seg_list:
                    mask = (time >= s) & (time <= e)

                    if not mask.any():
                        continue

                    for m in base_muscles:
                        if np.max(act_data[m][mask]) >= 1:
                            problematic_segs.append({
                            'subject': trial_name,
                            'side': side,
                            'start_time': s,
                            'end_time':float(e)
                            })
                            break
                        compiled_segs[subject][m].append(act_data[m][mask])

    return compiled_segs, problematic_segs

In [ ]:
muscles = [
    "addbrev_r", "addlong_r", "addmagDist_r", "addmagIsch_r", "addmagMid_r", "addmagProx_r",
    "bflh_r", "bfsh_r", "edl_r", "ehl_r", "fdl_r", "fhl_r", "gaslat_r", "gasmed_r",
    "glmax1_r", "glmax2_r", "glmax3_r", "glmed1_r", "glmed2_r", "glmed3_r",
    "glmin1_r", "glmin2_r", "glmin3_r", "grac_r", "iliacus_r", "perbrev_r", "perlong_r",
    "piri_r", "psoas_r", "recfem_r", "sart_r", "semimem_r", "semiten_r", "soleus_r", "tfl_r",
    "tibant_r", "tibpost_r", "vasint_r", "vaslat_r", "vasmed_r",

    "addbrev_l", "addlong_l", "addmagDist_l", "addmagIsch_l", "addmagMid_l", "addmagProx_l",
    "bflh_l", "bfsh_l", "edl_l", "ehl_l", "fdl_l", "fhl_l", "gaslat_l", "gasmed_l",
    "glmax1_l", "glmax2_l", "glmax3_l", "glmed1_l", "glmed2_l", "glmed3_l",
    "glmin1_l", "glmin2_l", "glmin3_l", "grac_l", "iliacus_l", "perbrev_l", "perlong_l",
    "piri_l", "psoas_l", "recfem_l", "sart_l", "semimem_l", "semiten_l",
    "soleus_l", "tfl_l", "tibant_l", "tibpost_l", "vasint_l", "vaslat_l", "vasmed_l"
]

act_segs, act_problem = activations_to_segs(
    muscles=muscles,                
    seg_times=all_segs,
    problem_trials=[],
    states_dir="/Users/briankeller/Desktop/GRFMuscleModel/Old_Young_Walking_Data/Results/SO", 
    states_filename="results_states.sto"
)

In [ ]:
print(len(act_problem), 'problematic segments (have activations greater than 1)')
for seg in act_problem:
    subj = seg.get('subject')
    time = seg.get('start_time')
    print(f'Trial:{subj}, start time:{time}')

In [ ]:
export_path = '/Users/briankeller/Desktop/GRFMuscleModel/Old_Young_Walking_Data/problematic_activation_segs'
with open(export_path, 'wb') as f:
    pickle.dump(act_problem, f)

In [ ]:
n_interp_points = 100
act_resampled = {}
time_resampled = None

for subj, subj_data in act_segs.items():
    act_resampled[subj] = {}
    for key, seg_list in subj_data.items():
        if len(seg_list) == 0:
            act_resampled[subj][key] = []
            continue
        resampled, t = interp_segments(seg_list, n_interp_points)
        act_resampled[subj][key] = resampled
        if time_resampled is None:
            t = np.asarray(t)
            time_resampled = t[0] if t.ndim == 2 else t

act_resampled["time_resampled"] = time_resampled

In [ ]:
base_muscles = sorted({m[:-2] for m in muscles if m.endswith(("_r", "_l"))})
act_compiled = {m: get_all_segments(act_resampled, m) for m in base_muscles}

In [ ]:
def plot_muscle_grid(
    resampled_muscles,
    time_resampled,
    muscle_keys,
    nrows=11,
    ncols=4,
    figsize=(15, 25),
    alpha=0.4,
    linewidth=2,
    plot_mean=True,
    mean_linewidth=3,
):
    """
    resampled_muscles[muscle] -> list or array of (N_segments, T)
    time_resampled[0] -> (T,) in [0,1]
    muscle_keys -> list of base muscle names (strings)
    """

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten()

    x = time_resampled * 100  # percent stance

    for i, ax in enumerate(axes):
        if i >= len(muscle_keys):
            ax.axis("off")
            continue

        key = muscle_keys[i]
        segments = resampled_muscles[key]

        # plot all segments
        for seg in segments:
            ax.plot(x, seg, linewidth=linewidth, color="#A2C7E7", alpha=alpha)

        # mean curve
        if plot_mean and len(segments) > 0:
            Y = np.asarray(segments)
            if Y.ndim == 2:
                ax.plot(x, Y.mean(axis=0), linewidth=mean_linewidth)

        # labels
        if i >= (nrows - 1) * ncols:
            ax.set_xlabel("Percent Normalized Stance", fontsize=12)
        if i % ncols == 0:
            ax.set_ylabel("Muscle Force (N)", fontsize=12)

        # auto title from key
        title = key.replace("_", " ").title()
        ax.set_title(title, fontsize=18)

        ax.tick_params(axis="x", labelsize=16)
        ax.tick_params(axis="y", labelsize=16)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_muscle_grid(
    resampled_muscles=act_compiled,
    time_resampled=act_resampled["time_resampled"],
    muscle_keys=base_muscles,
    nrows=11,
    ncols=4,
    figsize=(15, 25),
    alpha=0.25,
    linewidth=1.2,
    plot_mean=True
)

In [ ]:
left = 0
right = 0
for subjects, trials in act_segs.items():
    for trial_data in trials.values():
        left += len(trial_data['left'])
        right += len(trial_data['right'])
print(f'{left} left foot segments')
print(f'{right} right foot segments')
print(f'{left + right} segments total')

# Silder & Uhlrich Model Input / Output Visualization

Loads pre-processed, normalized, filtered segment pickles for both datasets and provides
grid plots of all model inputs (GRF x/y/z, COP x/y/z) and outputs (muscle forces, joint
contact forces). Use the **Filtering options** cell to restrict plots to specific subjects
or signal keys.

In [ ]:
import pickle

import numpy as np
import yaml

from grf_pipeline_utils.data_utils import flatten_to_muscle_dict, get_all_segments, plot_muscle_grid

repo_root    = os.path.abspath('../')
with open(os.path.join(repo_root, 'config.yaml')) as f:
    cfg = yaml.safe_load(f)

silder_dir  = os.path.join(repo_root, cfg['silder']['results']['processed'])
ulrich_dir  = os.path.join(repo_root, 'data/processed/Ulrich')

INPUT_KEYS  = cfg['signals']['inputs']   # ['grf_x','grf_y','grf_z','cop_x','cop_y','cop_z']
OUTPUT_KEYS = cfg['signals']['outputs']  # muscles + JRFs
MUSCLE_KEYS = [k for k in OUTPUT_KEYS if not k.startswith(('knee_', 'ankle_'))]
JRF_KEYS    = [k for k in OUTPUT_KEYS if k.startswith(('knee_', 'ankle_'))]

print('Inputs :', INPUT_KEYS)
print('Muscles:', MUSCLE_KEYS)
print('JRFs   :', JRF_KEYS)

## Load Processed Data

In [ ]:
# Silder — OA and YA cohorts saved separately by Silder_Batch_Processing
with open(os.path.join(silder_dir, 'Silder_OA_segs_normalized_filtered'), 'rb') as f:
    silder_oa = pickle.load(f)
with open(os.path.join(silder_dir, 'Silder_YA_segs_normalized_filtered'), 'rb') as f:
    silder_ya = pickle.load(f)

# Merge into one dict for whole-Silder plots
silder_all = {k: v for d in (silder_oa, silder_ya) for k, v in d.items() if k != 'time_resampled'}
silder_all['time_resampled'] = silder_oa['time_resampled']

oa_subjects = sorted(k for k in silder_oa if k != 'time_resampled')
ya_subjects = sorted(k for k in silder_ya if k != 'time_resampled')
print(f'Silder OA: {len(oa_subjects)} subjects — {sum(len(silder_oa[s].get("grf_y",[])) for s in oa_subjects)} segs')
print(f'Silder YA: {len(ya_subjects)} subjects — {sum(len(silder_ya[s].get("grf_y",[])) for s in ya_subjects)} segs')

In [ ]:
# Uhlrich dataset
with open(os.path.join(ulrich_dir, 'Ulrich_segs_normalized_filtered'), 'rb') as f:
    ulrich_all = pickle.load(f)

ulrich_subjects = sorted(k for k in ulrich_all if k != 'time_resampled')
print(f'Uhlrich: {len(ulrich_subjects)} subjects — {sum(len(ulrich_all[s].get("grf_y",[])) for s in ulrich_subjects)} segs')

## Filtering Options

Set these variables before running any plot cell to restrict what's shown.
`None` means "use all".

In [ ]:
# ── Filtering options ─────────────────────────────────────────────────────────
# Set to a list of subject keys to restrict plots; None = all subjects
SILDER_SUBJECTS  = None   # e.g. ['OA1', 'OA2', 'Y1']
ULRICH_SUBJECTS  = None   # e.g. ['Subject1', 'Subject3']

# Set to a list of signal keys to restrict input / output plots; None = all
PLOT_INPUT_KEYS  = None   # e.g. ['grf_x', 'grf_y', 'grf_z']
PLOT_MUSCLE_KEYS = None   # e.g. ['achilles', 'tibant', 'soleus']
PLOT_JRF_KEYS    = None   # e.g. ['knee_fx', 'knee_fy']


def _subset(seg_dict, subjects=None):
    """Return a copy of seg_dict restricted to the requested subjects."""
    all_subj = [k for k in seg_dict if k != 'time_resampled']
    keep = subjects if subjects is not None else all_subj
    out = {s: seg_dict[s] for s in keep if s in seg_dict}
    out['time_resampled'] = seg_dict['time_resampled']
    return out


def _taxis(seg_dict):
    t = np.asarray(seg_dict['time_resampled'])
    return (t[0] if t.ndim == 2 else t) * 100  # percent stance

## Model Inputs — Silder Dataset

Ground reaction forces (x = anterior-posterior, y = vertical, z = medial-lateral) and
center of pressure, normalized by body mass (N/kg or m).

In [ ]:
def plot_input_grid(seg_dict, keys=None, subjects=None,
                    seg_color='#A2C7E7', mean_color='#1A5EB6',
                    seg_alpha=0.35, seg_lw=1.2, mean_lw=3,
                    ncols=3, fig_w=5, fig_h=3.5, title_suffix=''):
    """
    Plot one subplot per input signal.  Each individual segment is drawn as a
    thin, semi-transparent line; the mean is drawn on top.

    Parameters
    ----------
    seg_dict  : processed segment dict (subjects → signal → list of arrays)
    keys      : signals to plot; None = all INPUT_KEYS
    subjects  : subject keys to include; None = all
    """
    plot_keys = keys if keys is not None else INPUT_KEYS
    data = _subset(seg_dict, subjects)
    t    = _taxis(data)

    ncols = min(ncols, len(plot_keys))
    nrows = int(np.ceil(len(plot_keys) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w * ncols, fig_h * nrows), sharex=True)
    axes = np.array(axes).flatten()

    labels = {
        'grf_x': 'GRF  Anterior-Posterior (N/kg)',
        'grf_y': 'GRF  Vertical (N/kg)',
        'grf_z': 'GRF  Medial-Lateral (N/kg)',
        'cop_x': 'COP  X (m)',
        'cop_y': 'COP  Y (m)',
        'cop_z': 'COP  Z (m)',
    }

    for i, key in enumerate(plot_keys):
        ax   = axes[i]
        segs = get_all_segments(data, key)
        for seg in segs:
            ax.plot(t, seg, color=seg_color, linewidth=seg_lw, alpha=seg_alpha)
        if len(segs) > 0:
            ax.plot(t, np.mean(segs, axis=0), color=mean_color, linewidth=mean_lw)
        ax.set_title(labels.get(key, key), fontsize=12)
        ax.tick_params(labelsize=10)
        if i >= (nrows - 1) * ncols:
            ax.set_xlabel('% Stance', fontsize=11)

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    fig.suptitle(f'Model Inputs{title_suffix}  (n={len(segs)} segs shown)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


plot_input_grid(silder_all, keys=PLOT_INPUT_KEYS, subjects=SILDER_SUBJECTS,
                title_suffix=' — Silder (OA + YA)')

## Model Outputs — Silder Dataset

Muscle forces and joint contact forces (knee + ankle), normalized by body mass (N/kg).

In [ ]:
silder_plot = _subset(silder_all, SILDER_SUBJECTS)
muscle_keys_plot = PLOT_MUSCLE_KEYS if PLOT_MUSCLE_KEYS is not None else MUSCLE_KEYS

silder_muscles = flatten_to_muscle_dict(silder_plot, muscle_keys_plot)
t_silder = silder_plot['time_resampled']

ncols = 4
nrows = int(np.ceil(len(muscle_keys_plot) / ncols))
plot_muscle_grid(
    silder_muscles, t_silder, muscle_keys_plot,
    nrows=nrows, ncols=ncols,
    figsize=(5 * ncols, 3.5 * nrows),
)

In [ ]:
jrf_keys_plot = PLOT_JRF_KEYS if PLOT_JRF_KEYS is not None else JRF_KEYS

silder_jrfs = flatten_to_muscle_dict(silder_plot, jrf_keys_plot)

ncols = 3
nrows = int(np.ceil(len(jrf_keys_plot) / ncols))
plot_muscle_grid(
    silder_jrfs, t_silder, jrf_keys_plot,
    nrows=nrows, ncols=ncols,
    figsize=(5 * ncols, 3.5 * nrows),
)

## Model Inputs — Uhlrich Dataset

In [ ]:
plot_input_grid(ulrich_all, keys=PLOT_INPUT_KEYS, subjects=ULRICH_SUBJECTS,
                title_suffix=' — Uhlrich',
                seg_color='#E7A9A2', mean_color='#9C2B00')

## Model Outputs — Uhlrich Dataset

In [ ]:
ulrich_plot = _subset(ulrich_all, ULRICH_SUBJECTS)
t_ulrich = ulrich_plot['time_resampled']

ulrich_muscles = flatten_to_muscle_dict(ulrich_plot, muscle_keys_plot)

ncols = 4
nrows = int(np.ceil(len(muscle_keys_plot) / ncols))
plot_muscle_grid(
    ulrich_muscles, t_ulrich, muscle_keys_plot,
    nrows=nrows, ncols=ncols,
    figsize=(5 * ncols, 3.5 * nrows),
)

In [ ]:
ulrich_jrfs = flatten_to_muscle_dict(ulrich_plot, jrf_keys_plot)

ncols = 3
nrows = int(np.ceil(len(jrf_keys_plot) / ncols))
plot_muscle_grid(
    ulrich_jrfs, t_ulrich, jrf_keys_plot,
    nrows=nrows, ncols=ncols,
    figsize=(5 * ncols, 3.5 * nrows),
)

## Overlay: Silder vs. Uhlrich

Blue = Silder, Red = Uhlrich. Individual segments are semi-transparent; dataset means are
drawn as solid thick lines. Only signals present in both datasets are plotted.

In [ ]:
def plot_overlay_grid(
    dict_a, dict_b,
    keys,
    label_a='Silder', label_b='Uhlrich',
    color_a='#A2C7E7', mean_a='#005A9C',
    color_b='#E7A9A2', mean_b='#9C2B00',
    seg_alpha_a=0.3, seg_alpha_b=0.15,
    seg_lw=1.2, mean_lw=2.5,
    ncols=3, fig_w=5, fig_h=3.5,
    ylabel='Force (N/kg)', title='',
):
    """
    Overlay two segment dicts on the same axes, one subplot per key.
    dict_a / dict_b must already be filtered to the desired subjects.
    """
    t_a = _taxis(dict_a)
    t_b = _taxis(dict_b)

    ncols = min(ncols, len(keys))
    nrows = int(np.ceil(len(keys) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w * ncols, fig_h * nrows))
    axes = np.array(axes).flatten()

    for i, key in enumerate(keys):
        ax = axes[i]
        segs_a = get_all_segments(dict_a, key)
        segs_b = get_all_segments(dict_b, key)

        for seg in segs_a:
            ax.plot(t_a, seg, color=color_a, linewidth=seg_lw, alpha=seg_alpha_a)
        for seg in segs_b:
            ax.plot(t_b, seg, color=color_b, linewidth=seg_lw, alpha=seg_alpha_b)

        if len(segs_a) > 0:
            ax.plot(t_a, np.mean(segs_a, axis=0), color=mean_a, linewidth=mean_lw,
                    label=f'{label_a} mean')
        if len(segs_b) > 0:
            ax.plot(t_b, np.mean(segs_b, axis=0), color=mean_b, linewidth=mean_lw,
                    label=f'{label_b} mean', linestyle='--')

        ax.set_title(key.replace('_', ' ').title(), fontsize=12)
        ax.tick_params(labelsize=10)
        if i < ncols:
            ax.legend(fontsize=8, loc='upper right')
        if i >= (nrows - 1) * ncols:
            ax.set_xlabel('% Stance', fontsize=11)
        if i % ncols == 0:
            ax.set_ylabel(ylabel, fontsize=11)

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    if title:
        fig.suptitle(title, fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


# GRF + COP overlay
input_keys_plot = PLOT_INPUT_KEYS if PLOT_INPUT_KEYS is not None else INPUT_KEYS
plot_overlay_grid(
    silder_plot, ulrich_plot,
    keys=input_keys_plot,
    ylabel='GRF (N/kg) / COP (m)',
    title='Model Inputs — Silder vs. Uhlrich',
)

In [ ]:
# Muscle force overlay (Silder vs. Uhlrich)
# Only muscles present in both datasets
shared_muscles = [k for k in muscle_keys_plot if k in ulrich_muscles]

plot_overlay_grid(
    silder_plot, ulrich_plot,
    keys=shared_muscles,
    ncols=4,
    ylabel='Muscle Force (N/kg)',
    title='Muscle Forces — Silder vs. Uhlrich',
)

In [ ]:
# JRF overlay (Silder vs. Uhlrich)
shared_jrfs = [k for k in jrf_keys_plot if k in ulrich_jrfs]

plot_overlay_grid(
    silder_plot, ulrich_plot,
    keys=shared_jrfs,
    ncols=3,
    ylabel='Joint Contact Force (N/kg)',
    title='Joint Contact Forces — Silder vs. Uhlrich',
)